In [1]:
import pandas as pd
import numpy as np

## Future rail stops

In [3]:
routeshapes = pd.read_csv('./output/future/rail/shapes.txt')
routeshapes.head()

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,L-PLR1-CLFD-WSMD,-33.781953,151.047190,1,0.000000
1,L-PLR1-CLFD-WSMD,-33.782527,151.046673,2,80.182320
2,L-PLR1-CLFD-WSMD,-33.782692,151.046507,3,104.651857
3,L-PLR1-CLFD-WSMD,-33.782793,151.046403,4,119.782310
4,L-PLR1-CLFD-WSMD,-33.782982,151.046214,5,147.692514


In [4]:
stopswithroutes = pd.read_csv('./Stops/StopsWithRoutes.csv')
stopswithroutes.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_t,parent_sta,wheelchair,platform_c,mode,route
0,228055,228055.0,Pacific Hwy After Murray St,-33.003610,151.685566,NaN,NaN,0.0,NaN,Bus,NaN
1,228054,228054.0,Pacific Hwy Opp Ntaba Rd,-33.006195,151.684612,NaN,NaN,0.0,NaN,Bus,NaN
2,287146,287146.0,Flint St At Oxford St,-33.392373,148.014977,NaN,NaN,0.0,NaN,Bus,NaN
3,228056,228056.0,Pacific Hwy At South St,-32.999847,151.685495,NaN,NaN,0.0,NaN,Bus,NaN
4,228051,228051.0,Pacific Hwy At Violet Town Rd,-33.010927,151.673754,NaN,NaN,0.0,NaN,Bus,NaN


In [5]:
stopswithroutes['mode'].unique()

array(['Bus', 'Train', 'Metro', 'LightRail', 'Regional', nan, 'Ferry',
       'Coach'], dtype=object)

In [6]:
stopswithroutes['route'].unique()

array([nan, 'T2-A-L-C', 'T2-C-L-A', 'M-1-L-T & M-2-B-E',
       'M-1-T-L & M-2-E-B', 'Parent', 'M-1-L-T', 'M-1-T-L',
       'M-3a-M-W & M-3b-M-P', 'M-3a-W-M & M-3b-P-M', 'M-3a-M-W & M-5-E-W',
       'M-6-T-M', 'M-6-M-T', 'M-3a-W-M & M-5-W-E', 'M-4a-K-T & M-4b-K-N',
       'M-4a-T-K & M-4b-N-K', 'M-5-W-E', 'M-5-E-W', 'L-1-W-C', 'L-1-C-W',
       'L-2-P-O', 'L-2-O-P', 'M-4a-K-T', 'M-4a-T-K', 'M-4b-K-N',
       'M-4b-N-K'], dtype=object)

In [7]:
# Extract stops with route information

print(len(stopswithroutes))
stopswithroutes = stopswithroutes.loc[stopswithroutes['route'].isnull()==False].reset_index(drop=True)
print(len(stopswithroutes))

45066
429


In [8]:
# stops.txt

stopswithroutes.columns = ['stop_id','stop_code','stop_name','stop_lat','stop_lon','location_type','parent_station','wheelchair_boarding','platform_code','mode','route']
stopstxt = stopswithroutes[['stop_id','stop_code','stop_name','stop_lat','stop_lon','location_type','parent_station','wheelchair_boarding','platform_code']] 
stopstxt['wheelchair_boarding'] = "0"
stopstxt['stop_id'] = stopstxt['stop_id'].astype(str)
stopstxt['location_type'] = stopstxt.apply(lambda x: '1' if x['stop_id'].startswith('P') else x['location_type'], axis=1)
# stopstxt['stop_id'] = stopstxt['stop_id'].apply(lambda x: int('20200711'+str(x)))
# stopstxt['stop_code'] = stopstxt['stop_code'].apply(lambda x: float('20200711'+str(x)))
stopstxt.to_csv('./output/future/rail/stops.txt', index=False)


/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """
/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

In [9]:
# Replace route with shape_id 

print(len(stopswithroutes))
stopswithroutes = stopswithroutes.loc[stopswithroutes['route']!='Parent']
print(len(stopswithroutes))
stopswithroutes['route'] = stopswithroutes['route'].str.split(' & ')
stopswithroutes = stopswithroutes.assign(shape_id=stopswithroutes['route']).explode('shape_id')
stopswithroutes.head()

429
316


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code,mode,route,shape_id
0,214074,214074.0,"Homebush Station, Platform 4",-33.866790,151.086383,NaN,PST1216,2.0,4,Train,[T2-A-L-C],T2-A-L-C
1,214077,214077.0,"Homebush Station, Platform 7",-33.867061,151.086342,NaN,PST1216,2.0,7,Train,[T2-C-L-A],T2-C-L-A
2,2193111,2193111.0,"Canterbury Station, Platform 1",-33.911974,151.118607,NaN,PST1395,2.0,1,Metro,"[M-1-L-T, M-2-B-E]",M-1-L-T
2,2193111,2193111.0,"Canterbury Station, Platform 1",-33.911974,151.118607,NaN,PST1395,2.0,1,Metro,"[M-1-L-T, M-2-B-E]",M-2-B-E
3,2193112,2193112.0,"Canterbury Station, Platform 2",-33.911903,151.118290,NaN,PST1395,2.0,2,Metro,"[M-1-T-L, M-2-E-B]",M-1-T-L


In [10]:
routeshapes['shape_id'].unique()

array(['L-PLR1-CLFD-WSMD', 'L-PLR1-WSMD-CLFD', 'L-PLR2-OLYP-WSUN',
       'L-PLR2-WSUN-OLYP', 'M-1-LVPL-TLWG', 'M-1-TLWG-LVPL',
       'M-2-BKTN-EPNG', 'M-2-EPNG-BKTN', 'M-3a-MBRA-WSIA',
       'M-3a-WSIA-MBRA', 'M-3b-MBRA-PRMT', 'M-3b-PRMT-MBRA',
       'M-4a-KGRH-TLWG', 'M-4a-TLWG-KGRH', 'M-4b-KGRH-NWST',
       'M-4b-NWST-KGRH', 'M-5-EPNG-WSIA', 'M-5-WSIA-EPNG',
       'M-6-MCTR-TLWG', 'M-6-TLWG-MCTR', 'T2-CTYC-WSAR', 'T2-WSAR-CTYC'],
      dtype=object)

In [11]:
len(routeshapes['shape_id'].unique())

22

In [12]:
stopswithroutes['shape_id'].unique()

array(['T2-A-L-C', 'T2-C-L-A', 'M-1-L-T', 'M-2-B-E', 'M-1-T-L', 'M-2-E-B',
       'M-3a-M-W', 'M-3b-M-P', 'M-3a-W-M', 'M-3b-P-M', 'M-5-E-W',
       'M-6-T-M', 'M-6-M-T', 'M-5-W-E', 'M-4a-K-T', 'M-4b-K-N',
       'M-4a-T-K', 'M-4b-N-K', 'L-1-W-C', 'L-1-C-W', 'L-2-P-O', 'L-2-O-P'],
      dtype=object)

In [13]:
len(stopswithroutes['shape_id'].unique())

22

In [14]:
shapedict = {'T2-A-L-C':'T2-WSAR-CTYC', 
             'T2-C-L-A':'T2-CTYC-WSAR', 
             'M-1-L-T':'M-1-LVPL-TLWG', 
             'M-2-B-E':'M-2-BKTN-EPNG', 
             'M-1-T-L':'M-1-TLWG-LVPL', 
             'M-2-E-B':'M-2-EPNG-BKTN',
             'M-3a-M-W':'M-3a-MBRA-WSIA', 
             'M-3b-M-P':'M-3b-MBRA-PRMT', 
             'M-3a-W-M':'M-3a-WSIA-MBRA', 
             'M-3b-P-M':'M-3b-PRMT-MBRA', 
             'M-5-E-W':'M-5-EPNG-WSIA',
             'M-6-T-M':'M-6-TLWG-MCTR', 
             'M-6-M-T':'M-6-MCTR-TLWG', 
             'M-5-W-E':'M-5-WSIA-EPNG', 
             'M-4a-K-T':'M-4a-KGRH-TLWG', 
             'M-4b-K-N':'M-4b-KGRH-NWST',
             'M-4a-T-K':'M-4a-TLWG-KGRH', 
             'M-4b-N-K':'M-4b-NWST-KGRH', 
             'L-1-W-C':'L-PLR1-WSMD-CLFD', 
             'L-1-C-W':'L-PLR1-CLFD-WSMD', 
             'L-2-P-O':'L-PLR2-WSUN-OLYP', 
             'L-2-O-P':'L-PLR2-OLYP-WSUN'}

In [15]:
stopswithroutes = stopswithroutes.replace({'shape_id': shapedict})
stopswithroutes.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code,mode,route,shape_id
0,214074,214074.0,"Homebush Station, Platform 4",-33.866790,151.086383,NaN,PST1216,2.0,4,Train,[T2-A-L-C],T2-WSAR-CTYC
1,214077,214077.0,"Homebush Station, Platform 7",-33.867061,151.086342,NaN,PST1216,2.0,7,Train,[T2-C-L-A],T2-CTYC-WSAR
2,2193111,2193111.0,"Canterbury Station, Platform 1",-33.911974,151.118607,NaN,PST1395,2.0,1,Metro,"[M-1-L-T, M-2-B-E]",M-1-LVPL-TLWG
2,2193111,2193111.0,"Canterbury Station, Platform 1",-33.911974,151.118607,NaN,PST1395,2.0,1,Metro,"[M-1-L-T, M-2-B-E]",M-2-BKTN-EPNG
3,2193112,2193112.0,"Canterbury Station, Platform 2",-33.911903,151.118290,NaN,PST1395,2.0,2,Metro,"[M-1-T-L, M-2-E-B]",M-1-TLWG-LVPL


In [16]:
stopswithroutes['shape_id'].unique()

array(['T2-WSAR-CTYC', 'T2-CTYC-WSAR', 'M-1-LVPL-TLWG', 'M-2-BKTN-EPNG',
       'M-1-TLWG-LVPL', 'M-2-EPNG-BKTN', 'M-3a-MBRA-WSIA',
       'M-3b-MBRA-PRMT', 'M-3a-WSIA-MBRA', 'M-3b-PRMT-MBRA',
       'M-5-EPNG-WSIA', 'M-6-TLWG-MCTR', 'M-6-MCTR-TLWG', 'M-5-WSIA-EPNG',
       'M-4a-KGRH-TLWG', 'M-4b-KGRH-NWST', 'M-4a-TLWG-KGRH',
       'M-4b-NWST-KGRH', 'L-PLR1-WSMD-CLFD', 'L-PLR1-CLFD-WSMD',
       'L-PLR2-WSUN-OLYP', 'L-PLR2-OLYP-WSUN'], dtype=object)

In [17]:
stopswithroutes.to_csv('./Stops/StopsWithShapeIds.csv', index=False)

Using this to get distance along routeshapes from qGIS (line_locate_point in attribute fields).

In [18]:
stopswithroutes = pd.read_csv('./Stops/StopsWithShapeIds_and_DistAlongRoutes.csv')
stopswithroutes.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_t,parent_sta,wheelchair,platform_c,mode,route,shape_id,distAlongR
0,214074,214074.0,"Homebush Station, Platform 4",-33.866790,151.086383,NaN,PST1216,2.0,4.0,Train,['T2-A-L-C'],T2-WSAR-CTYC,47466.726998
1,214077,214077.0,"Homebush Station, Platform 7",-33.867061,151.086342,NaN,PST1216,2.0,7.0,Train,['T2-C-L-A'],T2-CTYC-WSAR,18740.025561
2,2193111,2193111.0,"Canterbury Station, Platform 1",-33.911974,151.118607,NaN,PST1395,2.0,1.0,Metro,"['M-1-L-T', 'M-2-B-E']",M-1-LVPL-TLWG,18988.520953
3,2193111,2193111.0,"Canterbury Station, Platform 1",-33.911974,151.118607,NaN,PST1395,2.0,1.0,Metro,"['M-1-L-T', 'M-2-B-E']",M-2-BKTN-EPNG,8717.658718
4,2193112,2193112.0,"Canterbury Station, Platform 2",-33.911903,151.118290,NaN,PST1395,2.0,2.0,Metro,"['M-1-T-L', 'M-2-E-B']",M-1-TLWG-LVPL,57381.330269


In [19]:
stopswithroutes['stop_id'] = stopswithroutes['stop_id'].astype(int).astype(str)

In [20]:
stopswithroutes.duplicated().sum()

0

In [21]:
stopswithroutes = stopswithroutes[['stop_id','shape_id','distAlongR']].sort_values(['shape_id','distAlongR']).reset_index(drop=True)
stopswithroutes

,stop_id,shape_id,distAlongR
0,2056016162,L-PLR1-CLFD-WSMD,18.725992
1,2056016152,L-PLR1-CLFD-WSMD,1662.915933
2,2056016142,L-PLR1-CLFD-WSMD,3083.789287
3,2056016132,L-PLR1-CLFD-WSMD,3920.042285
4,2056016172,L-PLR1-CLFD-WSMD,4334.809440
...,...,...,...
429,204832,T2-WSAR-CTYC,55553.303718
430,204261,T2-WSAR-CTYC,57110.944955
431,2015121,T2-WSAR-CTYC,57676.204700
432,2015135,T2-WSAR-CTYC,58988.169314


In [22]:
stopswithroutes.to_csv('StopsByRoute_rail.csv', index=False)

In [23]:
stopswithroutes['shape_id'].value_counts()

T2-CTYC-WSAR        40
T2-WSAR-CTYC        34
M-1-LVPL-TLWG       34
M-1-TLWG-LVPL       34
M-3a-WSIA-MBRA      25
M-3a-MBRA-WSIA      25
M-2-BKTN-EPNG       23
M-2-EPNG-BKTN       23
L-PLR1-CLFD-WSMD    17
L-PLR1-WSMD-CLFD    17
M-4a-TLWG-KGRH      15
M-5-EPNG-WSIA       15
M-4a-KGRH-TLWG      15
M-5-WSIA-EPNG       15
M-3b-MBRA-PRMT      14
M-3b-PRMT-MBRA      14
M-6-MCTR-TLWG       13
M-6-TLWG-MCTR       13
M-4b-NWST-KGRH      12
L-PLR2-OLYP-WSUN    12
L-PLR2-WSUN-OLYP    12
M-4b-KGRH-NWST      12
Name: shape_id, dtype: int64